<a href="https://colab.research.google.com/github/pmolinari2910/Modelizado-de-IA/blob/main/Copia_de_Practica_Unidad_1_Sietemas_Expertos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Desafío Individual: Sistema de Gestión de Obra Inteligente

## Contexto del Problema
Una empresa constructora está desarrollando una torre de gran altura y necesita automatizar dos procesos críticos para garantizar la seguridad y la eficiencia operativa:
* Evaluación de Riesgos en Obra (Lógica Deductiva): Determinar si es seguro continuar con las tareas de altura o excavación basándose en sensores climáticos y estructurales.
* Planificación de Maquinaria Pesada (Satisfacción de Restricciones): Asignar equipos (grúas, excavadoras) a zonas específicas del predio respetando límites de seguridad y espacio físico.

**Misión A:** Diagnóstico de Seguridad con expertaDebes programar un motor de inferencia que reciba datos de sensores y devuelva el nivel de riesgo del sitio. Este sistema actúa como un Cerebro Lógico para evitar accidentes.

Reglas a Implementar:
* **Riesgo Crítico** (Paro de Obra):  Si la velocidad del viento es $> 60\ km/h$ o si se detectan grietas en el suelo de fundación.
* **Riesgo Moderado** (Precaución): Si la velocidad del viento está entre $40\ km/h$ y $60\ km/h$ o si hay humedad extrema en zonas de excavación.
* **Bajo Riesgo** (Operación Normal): Si los vientos son $< 40\ km/h$ y no hay alertas estructurales activas.

**Consigna Técnica:** Utiliza el parámetro salience para asegurar que la regla de Riesgo Crítico se evalúe con la máxima prioridad ante cualquier otra condición.El sistema debe imprimir el diagnóstico final y la orden de seguridad correspondiente.

**Misión B:** Ubicación de Equipos con python-constraint

Debes encontrar la distribución óptima de 3 máquinas pesadas en 3 zonas de trabajo distintas. El sistema debe "podar" las opciones que violen las normativas de seguridad.

Restricciones (Reglas de Oro):
* **Grúa Torre:** Solo puede ubicarse en la Zona_Estable (debido a la necesidad de una base de hormigón reforzada).
* **Excavadora:** No puede ingresar a la Zona_Estrecha debido a sus dimensiones.
* **Hormigonera:** No puede estar en la misma zona que la Grúa Torre para evitar congestión de camiones.
* **Exclusividad:** Cada zona solo puede albergar una máquina a la vez para evitar colisiones.Consigna Técnica:Define las variables (Máquinas) y el dominio (Zonas de la obra).

Aplica las funciones de restricción para que el motor de búsqueda encuentre la única configuración válida.

### Mision A

In [ ]:
!pip install experta
!pip install --upgrade frozendict

In [ ]:
from experta import *

class EstadoObra(Fact):
    """Datos capturados por sensores en el sitio"""
    pass

class MotorSeguridad(KnowledgeEngine):

    # REGLA 1: ALTO RIESGO (Prioridad 10)

    @Rule(OR(EstadoObra(viento=P(lambda v: v > 60)),
             EstadoObra(grietas=True)),
          salience=10)
    def riesgo_critico(self):
        print("\n--- DIAGNÓSTICO FINAL ---")
        print("RIESGO: ALTO")
        print("ORDEN: DETENER OBRA")
        self.halt()

    # REGLA 2: RIESGO MODERADO (Prioridad 5)
    @Rule(OR(EstadoObra(viento=P(lambda v: 40 <= v <= 60)),
             EstadoObra(humedad_extrema=True)),
          salience=5)
    def riesgo_moderado(self):
        print("\n--- DIAGNÓSTICO FINAL ---")
        print("RIESGO: MODERADO")
        print("ORDEN: REFORZAR LA ESTRUCTURA / PRECAUCIÓN")

    # REGLA 3: BAJO RIESGO (Prioridad 1)
    @Rule(EstadoObra(viento=P(lambda v: v < 40),
                     grietas=False,
                     humedad_extrema=False),
          salience=1)
    def bajo_riesgo(self):
        print("\n--- DIAGNÓSTICO FINAL ---")
        print("RIESGO: BAJO")
        print("ORDEN: NORMAL, SEGUIR CON LA CONSTRUCCIÓN")


engine = MotorSeguridad()
engine.reset()

#EJEMPLO
# Datos: viento de 65 km/h y presencia de grietas
datos = EstadoObra(viento=20, grietas=False, humedad_extrema=False)

engine.declare(datos)
engine.run()


--- DIAGNÓSTICO FINAL ---
RIESGO: BAJO
ORDEN: NORMAL, SEGUIR CON LA CONSTRUCCIÓN


### Mision B

In [ ]:
!pip install python-constraint

  Preparing metadata (setup.py) ... done
  Created wheel for python-constraint: filename=python_constraint-1.4.0-py2.py3-none-any.whl size=24061 sha256=bbd9c30de0d7b8a09440c43d99560dbb45ff019ff188afdf86a20d250202e352
  Stored in directory: /root/.cache/pip/wheels/c1/d2/3d/082849b61a9c6de02d4a7c8a402c224640f08d8a971307b92b
Successfully built python-constraint


In [ ]:
from constraint import *

def planificar_maquinaria():
    problem = Problem()
    # 1) Definir máquinas (variables) y zonas (dominios)
    maquinas = ["Grua_Torre", "Excavadora", "Hormigonera"]
    zonas= ["Estable", "Estrecha", "Comun"]

    problem.addVariables(maquinas, zonas)

    # 2) Restricciones
    # Restriccion 1: Exclusividad (1 máquina por zona)
    problem.addConstraint(AllDifferentConstraint())

    #Restriccion 2: Grúa torre unicamente en zona estable
    problem.addConstraint(lambda gt: gt == "Zona_Estable", ["Grua_Torre"])

    #Restriccion 3: Prohibido Excavadora en zona estrecha
    problem.addConstraint(lambda ex: ex != "Zona_Estrecha", ["Excavadora"])

    #Restriccion 4: Hormigonera NO puede estar con la Grúa

    problem.addConstraint(lambda ho, gt: ho != gt, ("Hormigonera", "Grua_Torre"))

    # 3) Resolución del problema

    soluciones = problem.getSolutions()
    print(f"Se encontraron {len(soluciones)} configuraciones seguras:")
    for i, sol in enumerate(soluciones, 1):
        print(f"Opción {i}: {sol}")

planificar_maquinaria()

Se encontraron 0 configuraciones seguras:


## Parámetros de Entrega y Evaluación
El entregable debe cumplir con lo siguiente:

* **Justificación Funcional:** Debes explicar en celdas de texto por qué el modelo de grafos y árboles es superior a una simple lista de if/else para este problema.
* **Documentación:** El código debe estar respaldado por una explicación de cómo opera el motor de inferencia en cada caso.

*Tener en cuenta que se evalua proceso y no resultado. Acordarse de justificar las elecciones en cada caso.*